# DeepSCF: a periodic polymer
Run the cells in order using a **CPU runtime**. The first cell downloads DeepSCF and the tutorial inputs.

Clone → install → build → preprocess → predict → non-SCF.

*Tested in Colab on 26 October 2026: Python 3.13.16, PyTorch 2.11.0 (CPU), sisl 0.20.0.*

## 1. Clone the repositories
Download DeepSCF and this tutorial, then check the required files.

In [ ]:
%%bash
set -e
cd /content
if [ ! -d DeepSCF ]; then
  git clone https://github.com/yhkimlab/DeepSCF.git
  git -C DeepSCF checkout --detach fe3c3c18052b51101e438dad2ffa06f94e77b5bb
fi

if [ ! -d tutorial ]; then
  git clone --depth 1 --branch main https://github.com/YHKlab-RGLee/tutorial.git tutorial
fi


In [ ]:
%%bash
set -e
cd /content/DeepSCF
# Check the tutorial snapshot.
[ "$(git rev-parse HEAD)" = fe3c3c18052b51101e438dad2ffa06f94e77b5bb ]
test -s requirements.txt
test -s src/siesta-4.1-b4/Obj/gfortran-system.make
git log -1 --format='%h %cs %s'
test -s /content/tutorial/deepscf/01/src/inputs/common.fdf
test -s /content/tutorial/deepscf/01/src/predict_polymer.py
echo "DeepSCF and tutorial inputs ready"


## 2. Install dependencies
Use Colab's Python and PyTorch. Install the additional packages, then preview the polymer.

In [ ]:
%pip install -q torch h5py hydra-core==1.3.2 omegaconf==2.3.0 "sisl[viz]" matplotlib

In [ ]:
%matplotlib inline
import sys, subprocess
import torch, h5py, numpy, hydra, omegaconf
import sisl, sisl.viz

print('Python:', sys.version.split()[0], '| PyTorch:', torch.__version__, '| sisl:', sisl.__version__)
geometry = sisl.get_sile('/content/DeepSCF/tutorial/1.preprocess/structures/examples/polymer.fdf').read_geometry()
geometry.plot(axes='xz', show_cell=False, backend='matplotlib').show()


## 3. Build and launch SIESTA
Build the bundled modified SIESTA, then check its startup log. No input is supplied, so an input error and a nonzero exit code are expected.

In [ ]:
%%bash
set -e
apt-get update -qq
apt-get install -y -qq build-essential binutils gfortran libblas-dev liblapack-dev
cd /content/DeepSCF/src/siesta-4.1-b4/Obj
sh ../Src/obj_setup.sh
cp gfortran-system.make arch.make
# Save the build log.
make -j2 > /content/siesta-build.log 2>&1 || { tail -80 /content/siesta-build.log; exit 1; }
test -x siesta
echo "SIESTA build OK"


In [ ]:
import subprocess, tempfile

with tempfile.TemporaryDirectory() as workdir:
    result = subprocess.run(
        ['/content/DeepSCF/src/siesta-4.1-b4/Obj/siesta'],
        stdin=subprocess.DEVNULL, capture_output=True, text=True,
        cwd=workdir, timeout=15,
    )
print(result.stdout)
print(result.stderr)
print('Exit code:', result.returncode, '(no input supplied)')


## 4. Preprocess: SIESTA → grids → HDF5
Use the repository's 24-atom C/H polymer, periodic along **z**. The cell is 20 × 20 × 10 Å; the grid is 80 × 80 × 40.

Here **X = (0, 0, ½)** in reciprocal coordinates. The bands are folded into the supplied 10 Å repeat cell.

Use the downloaded inputs to create a fresh run folder.

In [ ]:
from pathlib import Path
import sys, shutil, tempfile

ROOT = Path('/content/DeepSCF')
SRC = Path('/content/tutorial/deepscf/01/src')
PYTHON = sys.executable
SIESTA = ROOT / 'src/siesta-4.1-b4/Obj/siesta'
(ROOT / 'results').mkdir(exist_ok=True)
WORK = Path(tempfile.mkdtemp(prefix='polymer-', dir=ROOT / 'results'))
SCF, NSCF = WORK / 'scf/polymer', WORK / 'non_scf'
for folder, mode in [(SCF, 'scf'), (NSCF, 'non_scf')]:
    folder.mkdir(parents=True)
    for name in ['STRUCT.fdf', 'common.fdf', 'C.psf', 'H.psf']:
        shutil.copy(SRC / 'inputs' / name, folder / name)
    shutil.copy(SRC / 'inputs' / mode / 'RUN.fdf', folder / 'RUN.fdf')
print('Working directory:', WORK)
print((SCF / 'RUN.fdf').read_text())
print((SCF / 'common.fdf').read_text())
geometry = sisl.get_sile(str(SCF / 'RUN.fdf')).read_geometry()
geometry.plot(axes='xz', show_cell='box', backend='matplotlib').show()


Run the reference SCF calculation, then apply the **(R+S)×3** feature scaling. The reference density is used for evaluation, not as a network input.

In [ ]:
subprocess.run([PYTHON, str(SRC / 'run_siesta.py'), str(SCF),
                '--siesta', str(SIESTA)], check=True)
subprocess.run([PYTHON, str(ROOT / 'tutorial/1.preprocess/1.convert.py'),
                str(WORK / 'scf'), str(WORK / 'raw')], check=True)
subprocess.run([PYTHON, str(ROOT / 'tutorial/1.preprocess/3.collect.py'),
                str(WORK / 'raw'), str(ROOT / 'datasets/scale/scale_r_plus_s_x3.txt'),
                str(WORK / 'dataset')], check=True)


Inspect the density projected onto x–z and the Γ–X bands. Band energies use the SCF valence-band maximum as zero.

In [ ]:
sys.path.insert(0, str(SRC))
from analysis import plot_density, plot_bands, compare_energy

plot_density(SCF / 'Molecular.RHO', atomic=SCF / 'rhoatom');
plot_bands(SCF);


## 5. Predict: (R+S)×3
Download the pretrained archive and retain only the requested model (about 486 MB downloaded). No training dataset is needed.

In [ ]:
%%bash
set -euo pipefail
cd /content/DeepSCF
if [ ! -s pretrained/r_plus_s_x3/model.pt ]; then
  curl -fL --retry 3 https://github.com/yhkimlab/DeepSCF/releases/download/DeepSCF_v1/pretrained-models-v1.tar.gz \
    | tar --no-same-owner -xz pretrained/r_plus_s_x3/model.pt
fi
test -s pretrained/r_plus_s_x3/model.pt


Predict on CPU using the repository's model and configuration. The helper preserves the SCF cell and grid in the exported `polymer.RHO`.

Density errors use the full 3D grids; the plots show projections.

In [ ]:
subprocess.run([PYTHON, str(SRC / 'predict_polymer.py'), str(ROOT), str(WORK)], check=True)
PREDICTED = WORK / 'predict/polymer.RHO'
plot_density(SCF / 'Molecular.RHO', atomic=SCF / 'rhoatom', prediction=PREDICTED);


## 6. Evaluate: one-step non-SCF
Copy the predicted density to `In.RHO`. Use the same structure, pseudopotentials, mesh, and k points; do not copy an SCF density-matrix restart.

In [ ]:
shutil.copy(PREDICTED, NSCF / 'In.RHO')
print((NSCF / 'RUN.fdf').read_text())
print((NSCF / 'common.fdf').read_text())


Run one iteration and compare total energies and Γ–X bands. `SCF_NOT_CONV` is expected for this step.

In [ ]:
subprocess.run([PYTHON, str(SRC / 'run_siesta.py'), str(NSCF),
                '--siesta', str(SIESTA), '--non-scf'], check=True)
compare_energy(SCF, NSCF)
plot_bands(SCF, NSCF);


The fixed 0.25 Å grid follows the pretrained example; basis, mesh, and k-point convergence are not established here. The reference SCF is included for comparison, so the full workflow is not a speedup benchmark.

Logs: `stdout.txt` and `stderr.txt` in each run folder. Build log: `/content/siesta-build.log`.

[DeepSCF tutorial](https://github.com/yhkimlab/DeepSCF/blob/fe3c3c18052b51101e438dad2ffa06f94e77b5bb/tutorial/README.md) · [Polymer structure](https://github.com/yhkimlab/DeepSCF/blob/fe3c3c18052b51101e438dad2ffa06f94e77b5bb/tutorial/1.preprocess/structures/examples/polymer.fdf) · [sisl bands](https://sisl.readthedocs.io/en/latest/api/io/generated/sisl.io.siesta.bandsSileSiesta.html)
